In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
        print(os.path.join(dirname))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/plhalvorsen
/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset
/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/dyed-lifted-polyps
/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/dyed-lifted-polyps/dyed-lifted-polyps
/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/normal-z-line
/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/normal-z-line/normal-z-line
/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/dyed-resection-margins
/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/dyed-resection-margins/dyed-resection-margins
/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/normal-pylorus
/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/normal-pylorus/normal-pylorus
/kaggle/input

In [2]:
import shutil
from pathlib import Path
import random

random.seed(42)

# Original dataset root
SRC = Path("/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset")

# Classes you want
SELECTED_CLASSES = [
    "esophagitis",
    "polyps",
    "ulcerative-colitis"
]

# Output directory
DST = Path("/kaggle/working/kvasir_3class")
TRAIN_RATIO = 0.8

# Remove previous output
if DST.exists():
    shutil.rmtree(DST)

for cls in SELECTED_CLASSES:
    # Dataset has nested folders with same name
    class_dir = SRC / cls / cls

    images = list(class_dir.glob("*.*"))
    random.shuffle(images)

    split_idx = int(len(images) * TRAIN_RATIO)
    train_imgs = images[:split_idx]
    val_imgs = images[split_idx:]

    for subset, files in [("train", train_imgs), ("val", val_imgs)]:
        out_dir = DST / subset / cls
        out_dir.mkdir(parents=True, exist_ok=True)

        for f in files:
            shutil.copy(f, out_dir / f.name)

print("3-class dataset created successfully!")

3-class dataset created successfully!


In [3]:
from pathlib import Path

for subset in ["train", "val"]:
    print(f"\n{subset.upper()}")
    for cls in SELECTED_CLASSES:
        n = len(list((DST / subset / cls).glob("*")))
        print(f"{cls}: {n}")


TRAIN
esophagitis: 800
polyps: 800
ulcerative-colitis: 800

VAL
esophagitis: 200
polyps: 200
ulcerative-colitis: 200


In [4]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126 scikit-learn

Looking in indexes: https://download.pytorch.org/whl/cu126


In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Data transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Datasets
train_dataset = datasets.ImageFolder(
    "/kaggle/working/kvasir_3class/train",
    transform=transform
)

val_dataset = datasets.ImageFolder(
    "/kaggle/working/kvasir_3class/val",
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

print("Classes:", train_dataset.classes)

# Load pretrained SqueezeNet
model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT)

# Replace classifier for 3 classes
model.classifier[1] = nn.Conv2d(512, 3, kernel_size=1)
model.num_classes = 3
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Train
for epoch in range(10):
    model.train()
    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {running_loss/len(train_loader):.4f}")

Classes: ['esophagitis', 'polyps', 'ulcerative-colitis']
Downloading: "https://download.pytorch.org/models/squeezenet1_1-b8a52dc0.pth" to /root/.cache/torch/hub/checkpoints/squeezenet1_1-b8a52dc0.pth


100%|██████████| 4.73M/4.73M [00:00<00:00, 65.0MB/s]


Epoch 1, Loss: 0.9366
Epoch 2, Loss: 0.5233
Epoch 3, Loss: 0.4175
Epoch 4, Loss: 0.3843
Epoch 5, Loss: 0.3446
Epoch 6, Loss: 0.3505
Epoch 7, Loss: 0.3654
Epoch 8, Loss: 0.3675
Epoch 9, Loss: 0.2940
Epoch 10, Loss: 0.2711


In [6]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print("Validation Accuracy:", 100 * correct / total)

Validation Accuracy: 88.83333333333333


In [10]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torch.cuda.amp import GradScaler, autocast

CONFIG = {
    "train_dir"   : "/kaggle/working/kvasir_3class/train",
    "val_dir"     : "/kaggle/working/kvasir_3class/val",
    "num_classes" : 3,
    "epochs"      : 50,
    "batch_size"  : 128,
    "num_workers" : 4,
    "lr"          : 2e-3,
    "weight_decay": 1e-4,
    "save_path"   : "/kaggle/working/best_squeezenet.pth",
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(CONFIG["train_dir"], transform=train_transform)
val_ds   = datasets.ImageFolder(CONFIG["val_dir"],   transform=val_transform)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,
                          num_workers=CONFIG["num_workers"], pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False,
                          num_workers=CONFIG["num_workers"], pin_memory=True,
                          persistent_workers=True, prefetch_factor=2)

print(f"Classes : {train_ds.classes}")
print(f"Train   : {len(train_ds)} | Val: {len(val_ds)}")

device = torch.device("cuda")

model = models.squeezenet1_1(weights=models.SqueezeNet1_1_Weights.DEFAULT)
model.classifier[1] = nn.Conv2d(512, CONFIG["num_classes"], kernel_size=1)
model.num_classes = CONFIG["num_classes"]

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs via DataParallel")
    model = nn.DataParallel(model)

model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.Adam(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CONFIG["lr"],
    steps_per_epoch=len(train_loader),
    epochs=CONFIG["epochs"], pct_start=0.2,
)
scaler = GradScaler()

best_val_acc = 0.0

for epoch in range(CONFIG["epochs"]):
    t0 = time.time()

    model.train()
    tr_loss = tr_correct = tr_total = 0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            out  = model(imgs)
            loss = criterion(out, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        tr_loss    += loss.item()
        tr_correct += out.argmax(1).eq(lbls).sum().item()
        tr_total   += lbls.size(0)

    model.eval()
    val_loss = val_correct = val_total = 0
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs, lbls = imgs.to(device, non_blocking=True), lbls.to(device, non_blocking=True)
            with autocast():
                out  = model(imgs)
                loss = criterion(out, lbls)
            val_loss    += loss.item()
            val_correct += out.argmax(1).eq(lbls).sum().item()
            val_total   += lbls.size(0)

    tr_acc  = 100. * tr_correct  / tr_total
    val_acc = 100. * val_correct / val_total
    print(f"Epoch [{epoch+1:02d}/{CONFIG['epochs']}]  {time.time()-t0:.1f}s  "
          f"Train Loss: {tr_loss/len(train_loader):.4f}  Acc: {tr_acc:.2f}%  |  "
          f"Val Loss: {val_loss/len(val_loader):.4f}  Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
        torch.save(state, CONFIG["save_path"])
        print(f"  ✓ Saved best model (Val Acc: {best_val_acc:.2f}%)")

print(f"\nDone. Best Val Acc: {best_val_acc:.2f}%")

Classes : ['esophagitis', 'polyps', 'ulcerative-colitis']
Train   : 2400 | Val: 600
Using 2 GPUs via DataParallel


/tmp/ipykernel_57/1528204797.py:71: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_57/1528204797.py:83: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/tmp/ipykernel_57/1528204797.py:89: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()
/tmp/ipykernel_57/1528204797.py:99: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch [01/50]  21.4s  Train Loss: 0.7491  Acc: 71.96%  |  Val Loss: 0.5631  Acc: 85.33%
  ✓ Saved best model (Val Acc: 85.33%)
Epoch [02/50]  21.2s  Train Loss: 0.4960  Acc: 90.58%  |  Val Loss: 0.4573  Acc: 92.33%
  ✓ Saved best model (Val Acc: 92.33%)
Epoch [03/50]  21.1s  Train Loss: 0.4470  Acc: 93.12%  |  Val Loss: 0.4206  Acc: 95.83%
  ✓ Saved best model (Val Acc: 95.83%)
Epoch [04/50]  21.0s  Train Loss: 0.4221  Acc: 94.17%  |  Val Loss: 0.4427  Acc: 95.33%
Epoch [05/50]  21.2s  Train Loss: 0.4263  Acc: 94.50%  |  Val Loss: 0.3899  Acc: 95.50%
Epoch [06/50]  21.2s  Train Loss: 0.4130  Acc: 94.58%  |  Val Loss: 0.4127  Acc: 95.00%
Epoch [07/50]  21.3s  Train Loss: 0.9904  Acc: 68.21%  |  Val Loss: 1.0830  Acc: 33.33%
Epoch [08/50]  21.0s  Train Loss: 1.1656  Acc: 32.08%  |  Val Loss: 1.0986  Acc: 33.33%
Epoch [09/50]  21.5s  Train Loss: 1.0973  Acc: 32.54%  |  Val Loss: 1.0986  Acc: 33.33%
Epoch [10/50]  21.3s  Train Loss: 1.1007  Acc: 33.67%  |  Val Loss: 1.0986  Acc: 33.33%
Epo

**SEGMENTATION**

In [2]:
!pip install segmentation-models-pytorch albumentations -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01


In [8]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2, os
import numpy as np
from sklearn.model_selection import train_test_split

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE = "/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/polyps/polyps"

# ── Split filenames ───────────────────────────────────────────────────────────
all_fnames = sorted(os.listdir(f"{BASE}"))

train_files, valtest_files = train_test_split(all_fnames, test_size=0.3,  random_state=42)
val_files,   test_files    = train_test_split(valtest_files, test_size=0.5, random_state=42)

print(f"Train: {len(train_files)} | Val: {len(val_files)} | Test: {len(test_files)}")

# ── Dataset ───────────────────────────────────────────────────────────────────
class KvasirPolyp(Dataset):
    def __init__(self, fnames, base_dir, transform=None):
        self.fnames    = fnames
        self.base_dir  = base_dir
        self.transform = transform

    def __len__(self):
        return len(self.fnames)

    def __getitem__(self, idx):
        fname = self.fnames[idx]
        img   = cv2.imread(f"{self.base_dir}/images/{fname}")
        img   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(f"{self.base_dir}/masks/{fname}", cv2.IMREAD_GRAYSCALE)
        mask  = (mask > 127).astype(np.float32)

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img, mask = aug["image"], aug["mask"]

        return img, mask.unsqueeze(0)  # (3,H,W), (1,H,W)

# ── Augmentations ─────────────────────────────────────────────────────────────
train_tf = A.Compose([
    A.Resize(320, 320),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(p=0.4),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(320, 320),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# ── Dataloaders ───────────────────────────────────────────────────────────────
train_ds = KvasirPolyp(train_files, BASE, train_tf)
val_ds   = KvasirPolyp(val_files,   BASE, val_tf)
test_ds  = KvasirPolyp(test_files,  BASE, val_tf)

train_dl = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=8,  shuffle=False, num_workers=2, pin_memory=True)

# ── Sanity check ──────────────────────────────────────────────────────────────
imgs, masks = next(iter(train_dl))
print(f"Batch check — imgs: {imgs.shape} | masks: {masks.shape}")
# expected: torch.Size([16, 3, 320, 320]) torch.Size([16, 1, 320, 320])

# ── Model ─────────────────────────────────────────────────────────────────────
model = smp.DeepLabV3Plus(
    encoder_name="mobilenet_v2",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation=None,   # raw logits
)

# ── Loss ──────────────────────────────────────────────────────────────────────
dice_loss = smp.losses.DiceLoss(mode="binary")
bce_loss  = smp.losses.SoftBCEWithLogitsLoss()

def loss_fn(logits, masks):
    return dice_loss(logits, masks) + bce_loss(logits, masks)

# ── Metrics ───────────────────────────────────────────────────────────────────
def dice_score(pred_logits, targets, threshold=0.5):
    pred  = (torch.sigmoid(pred_logits) > threshold).float()
    inter = (pred * targets).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    return ((2 * inter + 1e-6) / (union + 1e-6)).mean().item()

def iou_score(pred_logits, targets, threshold=0.5):
    pred  = (torch.sigmoid(pred_logits) > threshold).float()
    inter = (pred * targets).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3)) - inter
    return ((inter + 1e-6) / (union + 1e-6)).mean().item()

# ── Training setup ────────────────────────────────────────────────────────────
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = model.to(device)
print(f"Training on: {device}")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=80, eta_min=1e-6)
scaler    = torch.cuda.amp.GradScaler()

EPOCHS    = 80
best_dice = 0.0
CKPT_PATH = "/kaggle/working/best_deeplabv3plus_mobilenetv2.pth"

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(EPOCHS):

    # train
    model.train()
    train_loss = 0.0
    for imgs, masks in train_dl:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(imgs)
            loss   = loss_fn(logits, masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()

    scheduler.step()
    train_loss /= len(train_dl)

    # val
    model.eval()
    val_dice = 0.0
    val_iou  = 0.0
    with torch.no_grad():
        for imgs, masks in val_dl:
            imgs, masks = imgs.to(device), masks.to(device)
            with torch.cuda.amp.autocast():
                logits = model(imgs)
            val_dice += dice_score(logits, masks)
            val_iou  += iou_score(logits, masks)

    val_dice /= len(val_dl)
    val_iou  /= len(val_dl)

    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Loss: {train_loss:.4f} | Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f}")

    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), CKPT_PATH)
        print(f"  ✓ Saved best model (Dice={best_dice:.4f})")

print(f"\nTraining done. Best Val Dice: {best_dice:.4f}")

# ── Test evaluation ───────────────────────────────────────────────────────────
print("\nRunning test set evaluation...")
model.load_state_dict(torch.load(CKPT_PATH))
model.eval()

test_dice = 0.0
test_iou  = 0.0
with torch.no_grad():
    for imgs, masks in test_dl:
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
        test_dice += dice_score(logits, masks)
        test_iou  += iou_score(logits, masks)

test_dice /= len(test_dl)
test_iou  /= len(test_dl)
print(f"Test Dice: {test_dice:.4f} | Test IoU: {test_iou:.4f}")

# ── ONNX Export ───────────────────────────────────────────────────────────────
print("\nExporting to ONNX...")
model.load_state_dict(torch.load(CKPT_PATH, map_location="cpu"))
model.eval().cpu()

dummy       = torch.randn(1, 3, 320, 320)
ONNX_PATH   = "/kaggle/working/deeplabv3plus_polyp.onnx"

torch.onnx.export(
    model,
    dummy,
    ONNX_PATH,
    opset_version=11,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input":  {0: "batch_size"},
        "output": {0: "batch_size"},
    }
)
print(f"ONNX saved → {ONNX_PATH}")

# ── Validate ONNX ─────────────────────────────────────────────────────────────
import onnxruntime as ort

sess = ort.InferenceSession(ONNX_PATH)
out  = sess.run(None, {"input": dummy.numpy()})
print(f"ONNX validation passed — output shape: {out[0].shape}")
# expected: (1, 1, 320, 320)

Train: 700 | Val: 150 | Test: 150


[ WARN:0@308.256] global loadsave.cpp:278 findDecoder imread_('/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/polyps/polyps/images/06c78a72-77c5-421e-8cc3-69ca951207dc.jpg'): can't open/read file: check file path/integrity
[ WARN:0@308.256] global loadsave.cpp:278 findDecoder imread_('/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/polyps/polyps/images/1015e8cf-735d-4e18-a221-229991f15cde.jpg'): can't open/read file: check file path/integrity
[ WARN:0@308.272] global loadsave.cpp:278 findDecoder imread_('/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/polyps/polyps/images/394a1183-22b9-4353-8117-1c6a636d28e0.jpg'): can't open/read file: check file path/integrity
[ WARN:0@308.272] global loadsave.cpp:278 findDecoder imread_('/kaggle/input/datasets/plhalvorsen/kvasir-v2-a-gastrointestinal-tract-dataset/polyps/polyps/images/5fd9427b-154e-4b2e-a967-310c20d24aac.jpg'): can't open/read file: check file

error: Caught error in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_57/1585530070.py", line 35, in __getitem__
    img   = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
cv2.error: OpenCV(4.13.0) /io/opencv/modules/imgproc/src/color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cvtColor'

